# WINGS3 — Production-grade eval (datasets + judges)

Run in JupyterLab workbench **wings3-demo**, project `my-first-model`.

**Red thread:** traces showed *what* happened; Act 3 showed a *toy* gate moved; these cells show a *reviewable* gate — a registered dataset, LLM judges with rationales, scores in MLflow.

**Say this first:** Llama 3.2 3B is the **agent**. Judges use hosted **gpt-oss-120b** (`JUDGE_*` from Secret `wings3-judge-llm`). That is the production-shaped split. Celebrate that scores now have **rationales** you can argue with. Hybrid scoring keeps `contains_expected` so a flaky judge row still has a cheap metric.

On stage, stop at each **SHOW:** comment. Live run is **v2 only**. If vLLM is cold, walk the SHOW cells and open a pre-logged Evaluation run.


## 0. Optional: git pull

JupyterLab root is this clone. Skip on stage if already current. Cluster must reach GitHub.


In [ ]:
# Optional: update from GitHub. Skip on stage if already current.
!git pull --ff-only


## 1b. Install deps if `langchain_core` is missing

Run this **before** the env cell on a fresh kernel (env imports langchain/mlflow). Skip if those imports already work. Re-run after a workbench restart — the venv is not on the PVC. `--extra-index-url` is required: the RHOAI 3.4 RHAI index has langgraph 1.x only and may not have `langchain-core`.


In [ ]:
# Kernel venv is not on the PVC. Skip if `import langchain_core` already works.
%pip install -r ../agent-tracing/requirements.txt --extra-index-url https://pypi.org/simple


## 1. Workbench env

Tracking URI is injected when the notebook has `opendatahub.io/mlflow-instance`. You still set `MLFLOW_WORKSPACE`. Experiment is **`wings3-agent-eval-prod`** so Act 3 numbers stay clean.


In [ ]:
import importlib
import json
import logging
import math
import os
import sys
import warnings
from pathlib import Path

import mlflow
from langchain_core.tools import tool
from mlflow.genai.scorers import Correctness, Guidelines, scorer

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

os.environ.setdefault("MLFLOW_WORKSPACE", "my-first-model")
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", "wings3-agent-eval-prod")
os.environ.setdefault("MAAS_API_KEY", "unused")
os.environ.setdefault("MAAS_MODEL", "llama-32-3b-instruct")
os.environ.setdefault(
    "MAAS_BASE_URL",
    "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1",
)

demo = Path("../agent-tracing").resolve()
if not (demo / "traced_agent.py").is_file():
    raise FileNotFoundError(f"expected traced_agent.py next to notebooks: {demo}")
sys.path.insert(0, str(demo))
traced_agent = importlib.import_module("traced_agent")
create_agent_graph = traced_agent.create_agent_graph
get_config_from_env = traced_agent.get_config_from_env

for k in (
    "MLFLOW_TRACKING_URI",
    "MLFLOW_WORKSPACE",
    "MLFLOW_K8S_INTEGRATION",
    "MLFLOW_TRACKING_AUTH",
    "MLFLOW_EXPERIMENT_NAME",
    "MAAS_MODEL",
    "MAAS_BASE_URL",
    "JUDGE_MODEL",
    "JUDGE_BASE_URL",
):
    print(f"{k}={os.environ.get(k)}")
print("JUDGE_API_KEY=" + ("set" if os.environ.get("JUDGE_API_KEY") else "MISSING"))


## 2. SHOW: golden JSONL

Eight calculator-only rows. `expected_answer` is the substring gate; `expected_facts` is what `Correctness` reads. First four rows are the Act 3 questions so the story continues.


In [ ]:
# SHOW: golden set lives in git — not an inline four-row list
GOLDEN_PATH = Path("../datasets/math_golden.jsonl").resolve()
if not GOLDEN_PATH.is_file():
    raise FileNotFoundError(GOLDEN_PATH)

EVAL_DATASET = []
with GOLDEN_PATH.open() as fh:
    for line in fh:
        line = line.strip()
        if line:
            EVAL_DATASET.append(json.loads(line))

print(f"Loaded {len(EVAL_DATASET)} rows from {GOLDEN_PATH.name}")
row = EVAL_DATASET[1]
print("Example:", row["inputs"]["user_message"])
print("contains_expected looks for:", row["expectations"]["expected_answer"])
print("Correctness looks for:", row["expectations"]["expected_facts"])


## 3. SHOW: register dataset

`create_dataset` + `merge_records` makes the golden set a first-class object in the MLflow **Datasets** tab. If `math_golden` already exists, drop its rows and merge from git — silent reuse kept `expected_response` beside `expected_facts` and Correctness refused to run. SQLite on this cluster is enough; this is not the Postgres/S3 production CR.


In [ ]:
uri = os.environ.get("MLFLOW_TRACKING_URI")
if not uri:
    raise RuntimeError("MLFLOW_TRACKING_URI is not set")
if not os.environ.get("MLFLOW_WORKSPACE"):
    raise RuntimeError("Set MLFLOW_WORKSPACE=my-first-model")

mlflow.set_tracking_uri(uri)
experiment = mlflow.set_experiment(
    os.environ.get("MLFLOW_EXPERIMENT_NAME", "wings3-agent-eval-prod")
)

DATASET_NAME = "math_golden"
try:
    eval_dataset = mlflow.genai.datasets.get_dataset(name=DATASET_NAME)
except Exception:
    eval_dataset = None

if eval_dataset is not None:
    # SHOW: drop stale rows, then merge git. Silent reuse kept expected_response
    # beside expected_facts and Correctness refused both.
    df = eval_dataset.to_df()
    rec_id = "dataset_record_id"
    if rec_id in df.columns and len(df):
        eval_dataset.delete_records(df[rec_id].tolist())
    eval_dataset = eval_dataset.merge_records(EVAL_DATASET)
    print(f"Refreshed {DATASET_NAME} from git ({len(EVAL_DATASET)} records)")
else:
    eval_dataset = mlflow.genai.datasets.create_dataset(
        name=DATASET_NAME,
        experiment_id=[experiment.experiment_id],
        tags={"wings3": "module-4", "kind": "golden"},
    )
    eval_dataset = eval_dataset.merge_records(EVAL_DATASET)
    print(f"Registered {DATASET_NAME} ({len(EVAL_DATASET)} records)")

print("Open MLflow → Datasets → math_golden")


## 4. SHOW: hybrid scorers

MLflow `openai:/…` is the **hosted OpenAI** provider — it always calls `api.openai.com` even if you set `OPENAI_BASE_URL`. Use **`hosted_vllm:/…`** plus `HOSTED_VLLM_API_BASE` (LiteLLM) so the judge hits hosted MaaS (`JUDGE_*` from Secret `wings3-judge-llm`). The agent stays on in-cluster 3B (`MAAS_*`). Keep `contains_expected` as the cheap safety net if a judge row flakes.



In [ ]:
@tool
def calculator(operation: str, a: float, b: float | None = None) -> str:
    """Arithmetic tool — same calculator as Act 2/3. For sqrt, pass only a."""
    ops = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y else "Error",
        "sqrt": lambda x, _: math.sqrt(x),
        "power": lambda x, y: x**y,
    }
    if operation not in ops:
        return f"Unknown operation {operation}"
    if operation != "sqrt" and b is None:
        return f"Error: {operation} needs two numbers a and b"
    result = ops[operation](a, b)
    return f"Result: {result}" if operation != "sqrt" else f"sqrt({a}) = {result}"


@scorer
def contains_expected(inputs: dict, outputs: str, expectations: dict) -> bool:
    if outputs is None or expectations is None:
        return False
    expected = str(expectations.get("expected_answer", ""))
    return expected.lower() in str(outputs).lower()


# SHOW: agent stays on MAAS_* (in-cluster 3B). Judges use JUDGE_* from Secret
# wings3-judge-llm. openai:/ hits api.openai.com. hosted_vllm:/ uses HOSTED_VLLM_*.
_judge_base = os.environ.get("JUDGE_BASE_URL") or "https://maas-rhdp.apps.maas.redhatworkshops.io/v1"
_judge_model = os.environ.get("JUDGE_MODEL") or "gpt-oss-120b"
_judge_key = (os.environ.get("JUDGE_API_KEY") or os.environ.get("HOSTED_VLLM_API_KEY") or "").strip()
if not _judge_key or _judge_key in {"unused", "REPLACE_ME"}:
    raise RuntimeError(
        "JUDGE_API_KEY is missing. oc apply -f manifests/secret-wings3-judge-llm.yaml, "
        "oc set env secret/wings3-judge-llm -n my-first-model JUDGE_API_KEY='…', "
        "then stop/start workbench wings3-demo."
    )
os.environ["HOSTED_VLLM_API_BASE"] = _judge_base
os.environ["HOSTED_VLLM_API_KEY"] = _judge_key
judge_model = f"hosted_vllm:/{_judge_model}"
print("Judge model:", judge_model)
print("HOSTED_VLLM_API_BASE:", os.environ["HOSTED_VLLM_API_BASE"])
print("JUDGE_API_KEY=set")
print("Other MaaS models: deepseek-r1-distill-qwen-14b, llama-scout-17b")

scorers = [
    contains_expected,
    Correctness(model=judge_model),
    Guidelines(
        name="numeric_and_clear",
        guidelines=[
            "The numeric result must appear as digits in the response.",
            "The response must state a single clear arithmetic result.",
        ],
        model=judge_model,
    ),
]
print("Scorers:", [getattr(s, "name", s) for s in scorers])


## 5. SHOW: `mlflow.genai.evaluate()`

v2 prompt from Act 3. This cell **defines** `run_eval`; the next cell calls the LLM (8 agent turns + 2 judges per row).


In [ ]:
V2_PROMPT = (
    "You are a precise math assistant. Always use the calculator tool for arithmetic. "
    "State the numeric result clearly in your answer."
)
print("=== v2 (judged) ===")
print(V2_PROMPT)

_agent = None


def get_agent():
    global _agent
    if _agent is None:
        _agent = create_agent_graph(
            get_config_from_env(), tools=[calculator], system_prompt=V2_PROMPT
        )
    return _agent


def predict_fn(user_message: str) -> str:
    try:
        result = get_agent().invoke(
            {"messages": [{"role": "user", "content": user_message}]}
        )
        return result["messages"][-1].content
    except Exception as exc:
        return f"Error: {exc}"


def run_eval() -> dict:
    global _agent
    _agent = None

    k8s = os.environ.get("MLFLOW_K8S_INTEGRATION", "").lower() == "true"
    if not k8s and not os.environ.get("MLFLOW_TRACKING_TOKEN"):
        raise RuntimeError("Run in the workbench or set MLFLOW_TRACKING_TOKEN")

    # One worker: MLflow 3.13's default pool deadlocks on first Databricks/Spark
    # import while several threads log traces at once. Interrupt will not unstick it.
    os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
    from mlflow.utils.databricks_utils import is_in_cluster, is_in_databricks_notebook

    is_in_cluster()
    is_in_databricks_notebook()
    mlflow.langchain.autolog()
    get_agent()
    print(f"Running evaluation: v2-judged ({len(EVAL_DATASET)} examples)")
    with mlflow.start_run(run_name="v2-judged"):
        # SHOW: dataset + predict_fn + hybrid scorers (substring + judges)
        result = mlflow.genai.evaluate(
            data=eval_dataset,
            predict_fn=predict_fn,
            scorers=scorers,
        )

    print("\nAggregated metrics:")
    for name, value in result.metrics.items():
        if isinstance(value, float):
            print(f"  {name}: {value:.2%}")
        else:
            print(f"  {name}: {value}")
    return result.metrics

print("Defined run_eval(). Next cell calls the LLM.")


## 6. Run v2 against the golden set

Skip this cell if rehearsal already logged `v2-judged` and vLLM is cold. Still walk the SHOW cells above.

If this cell sits with no output for more than a couple of minutes, **restart the kernel** (Interrupt is not enough) and re-run from the env cell. That hang is an MLflow thread deadlock, not a slow GPU.


In [ ]:
# Skip if v2-judged already exists in MLflow and the clock is tight.
metrics_v2 = run_eval()
metrics_v2


## 7. Metrics, then Evaluation UI

Standalone `/mlflow` → workspace `my-first-model` → experiment `wings3-agent-eval-prod`.

1. **Datasets** → `math_golden` (8 records).
2. **Evaluation** → run `v2-judged` — per-example `contains_expected`, `Correctness`, `numeric_and_clear`.
3. Open a row where substring and judge **disagree**, or a Fail with rationale, and read the judge text.


In [ ]:
def _fmt(metrics, key):
    if not metrics:
        return "—"
    val = metrics.get(key)
    if isinstance(val, float):
        return f"{val:.0%}"
    return val if val is not None else "—"


m = globals().get("metrics_v2") or {}
print(f"{'metric':<40} {'v2-judged':>10}")
print("-" * 52)
for key in sorted(m):
    print(f"{key:<40} {_fmt(m, key):>10}")

print()
print("Open MLflow → workspace my-first-model → experiment wings3-agent-eval-prod")
print("  Datasets → math_golden")
print("  Evaluation → v2-judged — pick a row and read the judge rationale")
